# 🧠 Day 2 — Deep Learning & Transformer Architecture
### CodeLucky · 12-Day Programme on Modern AI, Generative AI & Agentic Systems
**Module M2 · 6 Hours · 100% Hands-On · Runs Entirely in Google Colab (with GPU)**

> Yesterday (Day 1) scikit-learn did the learning behind one `.fit()` call. Today we open the engine: we build the kind of model that powers **ChatGPT, image generators, and modern translation** — the **neural network**, and its most important modern form, the **Transformer**.

**Goal of the day:** Understand — and build with your own hands — how a neural network learns, and how the Transformer (the "T" in ChatGPT / BERT / GPT) reads language.

| | |
|---|---|
| 🧩 **What we build** | A neural network from scratch, then a Transformer encoder block; we inspect real pre-trained BERT |
| 🗂️ **Data** | Iris flowers (first net) → MNIST digits (deeper net) → real sentences (Transformer) |
| 💻 **Where** | Google Colab with a free **GPU** |
| 🛠️ **Tools** | PyTorch · HuggingFace Transformers |


## 📖 Vocabulary you'll need
| Term | Meaning |
|---|---|
| Neural network | Layers of tiny units, each doing a simple calculation, loosely brain-inspired |
| Neuron | Takes numbers in, multiplies + adds them, passes one number out |
| Weight | A dial the network *learns* — how much to trust each input |
| Tensor | A grid of numbers (list, table, or cube) |
| PyTorch | The toolkit we use to build/train neural networks |
| GPU | A chip doing thousands of tiny maths ops at once — makes training fast |
| Transformer | The design behind ChatGPT — reads all words at once, decides what matters |


## 🧠 What Is a Neural Network, Really?
A single **neuron**: (1) takes numbers in, (2) multiplies each by a learned **weight** and sums them, (3) passes the result through a small "should I fire?" (activation) function. A network is many neurons stacked in **layers**.

**How it learns — the taste-test loop:**
1. **Guess** — predict with current weights
2. **Check** — measure how wrong (the "loss")
3. **Adjust** — nudge every weight toward less error (**gradient descent** — like a hiker walking downhill in fog)
4. **Repeat** thousands of times


## 🚀 Setup — turn on the Colab GPU
Menu: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

---
## 🟣 SESSION 1 — Tensors & Your First Neural Network
*(2 hours)* — Goal: understand tensors, then build and train a real neural network from scratch (flower classification).

### 1.1 Tensors: Just Grids of Numbers
- **0 directions** = a single number (scalar)
- **1 direction** = a list (vector)
- **2 directions** = a table (matrix)
- **3+ directions** = a cube of numbers (e.g. an image)

In [ ]:
import torch

a = torch.tensor(7)                        # scalar
b = torch.tensor([1.0, 2.0, 3.0])          # vector
c_ = torch.tensor([[1.0, 2.0], [3.0, 4.0]]) # matrix

print("a:", a, "| shape:", a.shape)
print("b:", b, "| shape:", b.shape)
print("c:\n", c_, "| shape:", c_.shape)

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])

print("Add:     ", x + y)
print("Multiply:", x * y)
print("Sum of x:", x.sum())

> Most beginner errors are *shape mismatches*. Whenever something breaks, print `.shape` first. Send a tensor to the GPU with `x = x.to(device)`.

### 1.2 Autograd — How PyTorch Learns the Adjust-Step
PyTorch automatically works out *which way is downhill* for every weight — you never do the calculus by hand.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
error = w ** 2          # a pretend error that depends on w
error.backward()        # PyTorch computes the gradient automatically

print("Current w:", w.item())
print("Gradient (slope) at w:", w.grad.item())   # 6.0

### 1.3 Meet the Data: Iris Flowers
150 flowers, 4 measurements each (petal/sepal length & width), 3 species. Task: 4 numbers in → 1 of 3 species out.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import torch

iris = load_iris()
X = iris.data
y = iris.target
print("Inputs shape:", X.shape)
print("First flower:", X[0], "-> species", y[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_test  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test  = torch.tensor(y_test,  dtype=torch.long).to(device)

print("Ready: training on", X_train.shape[0], "flowers")

### 1.4 Build Your First Neural Network
4 inputs → hidden layer → 3 outputs (one score per species).
- **`nn.Linear(4, 16)`** — layer of neurons, 4 in / 16 out, holds learnable weights
- **`nn.ReLU()`** — the "squish" (activation): negatives → 0, positives unchanged. Lets the network learn non-linear shapes.
- **`nn.Module`** — the PyTorch network blueprint (`__init__` lists layers, `forward` says how data flows)

In [ ]:
import torch.nn as nn

class FlowerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 3)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = FlowerNet().to(device)
print(model)

### 1.5 Train It — Watch It Get Smarter
- **Loss function** — `CrossEntropyLoss` for "pick one of several classes"
- **Optimizer** — `Adam`, `lr=0.01` is the learning rate (step size)

In [ ]:
import torch.optim as optim

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    predictions = model(X_train)                # 1. guess
    loss = loss_function(predictions, y_train)   # 2. check
    optimizer.zero_grad()                        # 3a. reset slopes
    loss.backward()                              # 3b. find downhill (autograd)
    optimizer.step()                             # 3c. nudge weights

    if (epoch + 1) % 20 == 0:
        print(f"Round {epoch+1:3d} | error (loss): {loss.item():.3f}")

### 1.6 Grade the Network

In [ ]:
model.eval()
with torch.no_grad():
    test_scores = model(X_test)
    predicted = test_scores.argmax(dim=1)
    accuracy = (predicted == y_test).float().mean()

print(f"Test accuracy: {accuracy.item()*100:.1f}%")   # usually ~95-100%

### 🧪 Try It Yourself
1. Build and train `FlowerNet` yourself, typing each layer.
2. Change the hidden layer size (`16` → `4` → `64`). Does accuracy/speed change?
3. Change `lr` (`0.01` → `0.1` → `0.001`). Watch the error overshoot or crawl.
4. What's the smallest network that still hits high accuracy?

✅ **Session 1 done.**


---
## 🟣 SESSION 2 — A Deeper Network That Sees Handwriting
*(2 hours)* — Goal: build a bigger, multi-layer network for MNIST digit recognition, and learn batching, dropout, and learning-rate tricks.

### 2.1 Meet the Data: MNIST Handwritten Digits
70,000 greyscale 28×28 images of digits 0–9. The "hello world" of deep learning.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

to_tensor = transforms.ToTensor()

train_data = datasets.MNIST(root="data", train=True,  download=True, transform=to_tensor)
test_data  = datasets.MNIST(root="data", train=False, download=True, transform=to_tensor)

print("Training images:", len(train_data))
print("One image shape:", train_data[0][0].shape)
print("Its label:", train_data[0][1])

### 2.2 Batching
We show the network small groups (**batches**, e.g. 64 images) instead of all 60,000 at once (too much memory) or one at a time (too slow/jumpy).

In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print("One batch of images:", images.shape)   # [64, 1, 28, 28]
print("One batch of labels:", labels.shape)   # [64]

### 2.3 Build the Digit Network — with Dropout
`nn.Dropout(0.2)` randomly switches off 20% of hidden neurons during training, forcing the network to learn robust, general patterns instead of memorising.

In [ ]:
class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.dropout(x)
        x = self.relu2(self.fc2(x))
        return self.fc3(x)

model = DigitNet().to(device)
print(model)

### 2.4 Train It — the Real Training Loop

In [ ]:
import torch.optim as optim

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()   # dropout ON

for epoch in range(3):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        predictions = model(images)
        loss = loss_function(predictions, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | average error: {avg:.3f}")

### 2.5 Grade It and Look at Its Mistakes

In [ ]:
model.eval()    # dropout OFF for testing
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        predicted = model(images).argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test accuracy: {correct/total*100:.2f}%")   # usually ~97%

In [ ]:
import matplotlib.pyplot as plt

img, true_label = test_data[0]
model.eval()
with torch.no_grad():
    guess = model(img.unsqueeze(0).to(device)).argmax(dim=1).item()

plt.imshow(img.squeeze(), cmap="gray")
plt.title(f"Model says: {guess}   |   Truth: {true_label}")
plt.axis("off")
plt.show()

### 🧪 Try It Yourself
1. Build and train `DigitNet`, report accuracy.
2. Remove dropout and retrain — does accuracy change?
3. Change `lr` to `0.1`, watch it misbehave; set back to `0.001`.
4. Show 5 test images with guess vs. truth; find one it gets wrong.

✅ **Session 2 done.**


---
## 🟣 SESSION 3 — Inside the Transformer (ChatGPT's Engine)
*(2 hours)* — Goal: understand **attention**, build a Transformer encoder block by hand, and inspect a real pre-trained BERT model.

### 3.1 The Problem Transformers Solve
Language is a *sequence*, and meaning depends on context: "I sat on the river **bank**" vs "I deposited cash at the **bank**." The Transformer lets every word look at every other word and decide which ones matter — this is **attention**.

### 3.2 Attention, Explained Simply
Three roles per word:
- **Query** — what this word is looking for
- **Key** — what each word offers as a label
- **Value** — the actual meaning each word contributes

Each word's Query is compared to every word's Key → attention scores → blend the Values. **Multi-head** attention = several attention passes in parallel, each tracking a different kind of relationship. **Positional encoding** = a tag telling the model each word's position (since attention alone loses word order).

### 3.3 Build a Self-Attention Block by Hand

In [ ]:
import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.query = nn.Linear(embed_size, embed_size)
        self.key   = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (x.shape[-1] ** 0.5)
        weights = self.softmax(scores)

        output = weights @ V
        return output, weights

fake_sentence = torch.rand(1, 5, 8)   # [1 sentence, 5 words, 8 numbers each]
attn = SelfAttention(embed_size=8)
out, weights = attn(fake_sentence)

print("Output shape:", out.shape)                 # [1, 5, 8]
print("Attention weights shape:", weights.shape)  # [1, 5, 5]

### 3.4 The Full Encoder Block
Wraps attention with a small feed-forward network and two stabilisers: **residual connections** (add the input back so nothing is forgotten) and **layer normalization** (keeps numbers in a healthy range).

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.attention = SelfAttention(embed_size)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size),
        )

    def forward(self, x):
        attn_out, _ = self.attention(x)
        x = self.norm1(x + attn_out)
        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)
        return x

block = EncoderBlock(embed_size=8)
result = block(fake_sentence)
print("Encoder block output shape:", result.shape)

> Stack 6, 12, or 24 of these `EncoderBlock`s, train on enormous text, and you have the architecture behind BERT and (with a small variation) ChatGPT.

### 3.5 Inspect a REAL Transformer: BERT

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

sentence = "I sat on the river bank."
tokens = tokenizer(sentence, return_tensors="pt")

print("Token IDs:", tokens["input_ids"])
print("Pieces:", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

In [ ]:
total_params = sum(p.numel() for p in bert.parameters())
print(f"BERT has {total_params:,} learned weights")   # ~110 million

print(bert)   # scroll through: 12 encoder layers, same shape as EncoderBlock above

In [ ]:
import torch
with torch.no_grad():
    output = bert(**tokens)

print("Meaning vectors shape:", output.last_hidden_state.shape)
# [1, 7, 768] -> 1 sentence, 7 tokens, 768 context-aware numbers per token

> **The big reveal:** BERT is 12 of the `EncoderBlock`s you built by hand — trained at scale. It isn't magic.

### 🧪 Try It Yourself
1. Build `SelfAttention` and `EncoderBlock` yourself, run the fake sentence through.
2. Tokenize a sentence of your own. Did any word split into pieces?
3. Run "river bank" vs "money bank" through BERT and compare.
4. Count encoder layers in `print(bert)`; find self-attention and feed-forward inside one layer.

✅ **Session 3 done.**


---
## 🏁 What You Built Today
- 🧮 Tensors: grids of numbers, shapes must line up
- 🧠 A neural network (FlowerNet) built and trained from scratch
- 🔢 A deeper network (DigitNet) with batching, dropout, learning rate tricks
- 🔤 Attention understood: Query · Key · Value
- 🏗️ A hand-built Transformer encoder block, and a real 110M-parameter BERT inspected

## 📚 One-Page Cheat-Sheet

In [ ]:
# THE DAY 2 PATTERNS

# 1. SET THE DEVICE
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. DEFINE A NETWORK
import torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(IN, HIDDEN)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(HIDDEN, OUT)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = Net().to(device)

# 3. THE TRAINING LOOP (guess -> check -> adjust)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(EPOCHS):
    pred = model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 4. EVALUATE
model.eval()
with torch.no_grad():
    acc = (model(X_test).argmax(1) == y_test).float().mean()

# 5. LOAD A REAL TRANSFORMER
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")
out = bert(**tok("Hello world", return_tensors="pt"))

### ✅ Day 2 Self-Check
- [ ] I turned on the Colab GPU and confirmed `torch.cuda.is_available()`
- [ ] I can explain what a tensor is and why `.shape` matters
- [ ] I understand guess → check → adjust
- [ ] I built and trained FlowerNet from scratch
- [ ] I know hidden layer, ReLU, loss, optimizer
- [ ] I built DigitNet and understand batching
- [ ] I can explain dropout
- [ ] I understand attention (Query/Key/Value)
- [ ] I built a self-attention layer and an encoder block by hand
- [ ] I loaded BERT and saw it's 12 of my encoder blocks, trained at scale

---
### 🚀 Coming Up — Day 3: LLMs, Prompting & Real APIs
*We finish the Transformer story, then start talking to real LLMs through local (Ollama) and cloud APIs — the bedrock skill for everything that follows.*

**CodeLucky · Faculty Development Programme**
